# Heart Disease Prediction on the UCI Cleveland Dataset
### Decision Tree vs Naive Bayes vs KNN vs ANN (Google Colab)

**Data source (official):** UCI Machine Learning Repository, *Heart Disease* dataset (id 45), https://archive.ics.uci.edu/dataset/45/heart+disease
Licence: CC BY 4.0. Cleveland Clinic file: 303 patients, 13 clinical features, target `num` (0 = no disease, 1-4 = disease).

This is a **different dataset** from the 70,000-record `cardio_train.csv`, so the pipeline is adapted for a small dataset with missing values (imputation, one-hot encoding, repeated cross-validation).

Run the cells **one by one from top to bottom** (Shift + Enter). 

## Step 1: Import libraries

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, cross_val_predict, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve, classification_report)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Libraries loaded")

## Step 2: Load the dataset (your `heartdisease.zip` from UCI)
Click **Choose files** and upload `heartdisease.zip` (or `processed.cleveland.data` directly). The cell unzips it and reads the **Cleveland** file `processed.cleveland.data` (303 patients), the standard file used for this dataset.  
If you cancel the upload, it downloads the same file from the official UCI website instead.

In [ ]:
import os, zipfile
from google.colab import files

COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
        "exang", "oldpeak", "slope", "ca", "thal", "num"]
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

up = files.upload()                      # choose heartdisease.zip
path = None
for name in up:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall("heartdisease")
        path = os.path.join("heartdisease", "processed.cleveland.data")
    elif "cleveland" in name.lower():
        path = name

if path and os.path.exists(path):
    df_raw = pd.read_csv(path, header=None, names=COLS, na_values="?")
    print("Loaded from your file:", path)
else:
    df_raw = pd.read_csv(URL, header=None, names=COLS, na_values="?")
    print("Loaded from the official UCI link")
print(df_raw.shape)

## Step 3: Inspect the data

In [ ]:
display(df_raw.head())
print("Shape:", df_raw.shape)
print("\nMissing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print("\nDuplicate rows:", df_raw.duplicated().sum())
display(df_raw.describe().T)
print("\nOriginal target values (num):")
print(df_raw["num"].value_counts().sort_index())

## Step 4: Data cleaning
- `?` was already converted to missing values (`NaN`): only `ca` and `thal` have a few.
- The target `num` is converted to **binary**: 0 = no heart disease, 1 = heart disease (original 1, 2, 3, 4), which is the standard use of this file.
- Missing values are not dropped; they are **imputed inside the model pipeline** so that no information from the test data leaks into training.

In [ ]:
df = df_raw.drop_duplicates().copy()
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

# make sure every column is numeric
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Rows:", len(df))
print("Class balance (0 = no disease, 1 = disease):")
print(df["target"].value_counts())
print("\nMissing values:", df.isna().sum()[df.isna().sum() > 0].to_dict())

## Step 5: Feature groups and encoding plan
| Group | Columns | Treatment |
|---|---|---|
| Numeric | age, trestbps, chol, thalach, oldpeak, ca | median imputation + standard scaling |
| Categorical (no natural order) | cp, restecg, slope, thal | most-frequent imputation + one-hot encoding |
| Binary | sex, fbs, exang | most-frequent imputation, kept as 0/1 |

In [ ]:
NUM = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
CAT = ["cp", "restecg", "slope", "thal"]
BIN = ["sex", "fbs", "exang"]
FEATURES = NUM + CAT + BIN
X, y = df[FEATURES], df["target"]

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT),
        ("bin", SimpleImputer(strategy="most_frequent"), BIN),
    ])

def make_pipe(clf):
    return Pipeline([("prep", make_preprocessor()), ("clf", clf)])

print("Features:", len(FEATURES), "| samples:", len(X))

## Step 6: Exploratory data analysis and risk factors

In [ ]:
labels = {0: "No disease", 1: "Disease"}
fig, ax = plt.subplots(2, 3, figsize=(16, 9))

ax[0, 0].bar([labels[i] for i in [0, 1]], y.value_counts().sort_index().values, color=["#4c78a8", "#e45756"])
ax[0, 0].set_title("Class distribution")

for c, col in [(0, "#4c78a8"), (1, "#e45756")]:
    ax[0, 1].hist(df[df.target == c].age, bins=15, alpha=.65, color=col, label=labels[c])
ax[0, 1].set_title("Age by class"); ax[0, 1].legend()

for c, col in [(0, "#4c78a8"), (1, "#e45756")]:
    ax[0, 2].hist(df[df.target == c].thalach, bins=15, alpha=.65, color=col, label=labels[c])
ax[0, 2].set_title("Maximum heart rate (thalach) by class"); ax[0, 2].legend()

rate = df.groupby("cp")["target"].mean() * 100
ax[1, 0].bar(rate.index.astype(int).astype(str), rate.values, color="#72b7b2")
ax[1, 0].set_title("Disease rate (%) by chest pain type"); ax[1, 0].set_xlabel("1 typical, 2 atypical, 3 non-anginal, 4 asymptomatic")

rate = df.groupby("exang")["target"].mean() * 100
ax[1, 1].bar(["No", "Yes"], rate.values, color="#f58518")
ax[1, 1].set_title("Disease rate (%) by exercise-induced angina")

rate = df.groupby("ca")["target"].mean() * 100
ax[1, 2].bar(rate.index.astype(int).astype(str), rate.values, color="#b279a2")
ax[1, 2].set_title("Disease rate (%) by number of major vessels (ca)")
plt.tight_layout(); plt.show()

corr = df[FEATURES + ["target"]].corr()
plt.figure(figsize=(8.5, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=60, ha="right"); plt.yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6.5)
plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

print("Correlation with the target:")
print(corr["target"].drop("target").sort_values(key=abs, ascending=False).round(3))

## Step 7: Train-test split and cross-validation setup
- **80 / 20 stratified hold-out split** (about 242 train, 61 test).
- Because the test set is tiny, results from one split are noisy. So models are also compared with **5-fold cross-validation repeated 5 times** (25 evaluations) on the full data.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", Xtr.shape, " Test:", Xte.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

## Step 8: Hyper-parameter tuning (Decision Tree and KNN)
Grid search with 5-fold cross-validation on the training set only.

In [ ]:
dt_search = GridSearchCV(
    make_pipe(DecisionTreeClassifier(random_state=SEED)),
    {"clf__max_depth": [2, 3, 4, 5, 6], "clf__min_samples_leaf": [2, 5, 10, 15]},
    cv=cv, scoring="accuracy", n_jobs=-1).fit(Xtr, ytr)
print("Best Decision Tree:", dt_search.best_params_, "CV acc =", round(dt_search.best_score_, 4))

knn_search = GridSearchCV(
    make_pipe(KNeighborsClassifier()),
    {"clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21], "clf__weights": ["uniform", "distance"]},
    cv=cv, scoring="accuracy", n_jobs=-1).fit(Xtr, ytr)
print("Best KNN:", knn_search.best_params_, "CV acc =", round(knn_search.best_score_, 4))

## Step 9: Define the four classifiers
The ANN is small (16, 8) with L2 regularisation because there are only about 240 training rows.

In [ ]:
models = {
    "Decision Tree": dt_search.best_estimator_,
    "Naive Bayes":   make_pipe(GaussianNB()),
    "KNN":           knn_search.best_estimator_,
    "ANN (MLP)":     make_pipe(MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu", alpha=0.01,
                                             max_iter=2000, random_state=SEED)),
}
print(list(models))

## Step 10: Repeated cross-validation (5 folds x 5 repeats, full data)
This is the most reliable comparison for a small dataset.

In [ ]:
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rows = []
for name, m in models.items():
    r = cross_validate(m, X, y, cv=rcv, scoring=scoring, n_jobs=-1)
    rows.append({"Model": name,
                 "Accuracy": f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
                 "Precision": f"{r['test_precision'].mean():.3f} ± {r['test_precision'].std():.3f}",
                 "Recall": f"{r['test_recall'].mean():.3f} ± {r['test_recall'].std():.3f}",
                 "F1-Score": f"{r['test_f1'].mean():.3f} ± {r['test_f1'].std():.3f}",
                 "ROC-AUC": f"{r['test_roc_auc'].mean():.3f} ± {r['test_roc_auc'].std():.3f}",
                 "_acc": r["test_accuracy"].mean(), "_f1": r["test_f1"].mean(), "_rec": r["test_recall"].mean()})
cv_table = pd.DataFrame(rows).set_index("Model")
display(cv_table.drop(columns=["_acc", "_f1", "_rec"]))

## Step 11: Train on the training set and test on the hold-out test set

In [ ]:
results, cms, probas, preds = {}, {}, {}, {}
for name, m in models.items():
    t0 = time.time(); m.fit(Xtr, ytr); fit_t = time.time() - t0
    pred = m.predict(Xte); proba = m.predict_proba(Xte)[:, 1]
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    results[name] = {"Accuracy": accuracy_score(yte, pred), "Precision": precision_score(yte, pred),
                     "Recall": recall_score(yte, pred), "F1-Score": f1_score(yte, pred),
                     "ROC-AUC": roc_auc_score(yte, proba), "Specificity": tn / (tn + fp),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp, "Train time (s)": fit_t,
                     "Train accuracy": accuracy_score(ytr, m.predict(Xtr))}
    cms[name], probas[name], preds[name] = np.array([[tn, fp], [fn, tp]]), proba, pred

results_df = pd.DataFrame(results).T
display(results_df.round(4))
print("Note: the test set has only", len(yte), "patients, so one extra correct prediction changes accuracy by about 1.6 points.")

## Step 12: Performance comparison chart (hold-out test and repeated CV)

In [ ]:
names = list(models)
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

w = 0.2
for i, mt in enumerate(metrics):
    vals = [results[n][mt] for n in names]
    bars = ax[0].bar(np.arange(4) + i * w, vals, w, label=mt)
    for b, v in zip(bars, vals):
        ax[0].text(b.get_x() + b.get_width() / 2, v + .005, f"{v:.2f}", ha="center", fontsize=7, rotation=90)
ax[0].set_xticks(np.arange(4) + 1.5 * w); ax[0].set_xticklabels(names)
ax[0].set_title("Hold-out test set (61 patients)"); ax[0].set_ylim(.5, 1.0); ax[0].legend(ncol=4, loc="upper center", fontsize=8)

cv_cols = {"Accuracy": "Accuracy", "Precision": "Precision", "Recall": "Recall", "F1-Score": "F1-Score"}
for i, mt in enumerate(metrics):
    vals = [float(cv_table.loc[n, mt].split(" ")[0]) for n in names]
    bars = ax[1].bar(np.arange(4) + i * w, vals, w, label=mt)
    for b, v in zip(bars, vals):
        ax[1].text(b.get_x() + b.get_width() / 2, v + .005, f"{v:.2f}", ha="center", fontsize=7, rotation=90)
ax[1].set_xticks(np.arange(4) + 1.5 * w); ax[1].set_xticklabels(names)
ax[1].set_title("Repeated 5-fold CV x 5 (full data, mean)")
plt.tight_layout(); plt.show()

## Step 13: Confusion matrices
Top row: hold-out test set. Bottom row: pooled predictions from 5-fold cross-validation on all 303 patients (more reliable).

In [ ]:
cv_cms = {}
for n in names:
    p = cross_val_predict(models[n], X, y, cv=cv, n_jobs=-1)
    cv_cms[n] = confusion_matrix(y, p)

fig, axs = plt.subplots(2, 4, figsize=(17, 8))
for row, (title, store) in enumerate([("Hold-out", cms), ("5-fold CV (all data)", cv_cms)]):
    for a, n in zip(axs[row], names):
        a.imshow(store[n], cmap="Blues")
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{store[n][i, j]}", ha="center", va="center", fontsize=13, fontweight="bold")
        a.set_xticks([0, 1]); a.set_yticks([0, 1])
        a.set_xticklabels(["No disease", "Disease"]); a.set_yticklabels(["No disease", "Disease"])
        a.set_xlabel("Predicted"); a.set_ylabel("Actual"); a.set_title(f"{n}\n({title})")
plt.tight_layout(); plt.show()

for n in names:
    print("=" * 60, "\n", n, "(hold-out)")
    print(classification_report(yte, preds[n], target_names=["No disease", "Disease"], digits=3))

## Step 14: ROC curves (hold-out test set)

In [ ]:
plt.figure(figsize=(6, 5))
for n in names:
    fpr, tpr, _ = roc_curve(yte, probas[n])
    plt.plot(fpr, tpr, lw=1.8, label=f"{n} (AUC {results[n]['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves"); plt.legend(); plt.tight_layout(); plt.show()

## Step 15: Feature importance
1. Decision Tree (Gini) importance on the encoded features.
2. Permutation importance on the original columns for all four models (drop in accuracy when a column is shuffled, 30 repeats). With only 61 test rows this is noisy, so look at the broad ranking only.

In [ ]:
dt_pipe = models["Decision Tree"]
enc_names = dt_pipe.named_steps["prep"].get_feature_names_out()
imp_dt = pd.Series(dt_pipe.named_steps["clf"].feature_importances_, index=enc_names).sort_values()

perm = {}
for n in names:
    r = permutation_importance(models[n], Xte, yte, n_repeats=30, random_state=SEED,
                               scoring="accuracy", n_jobs=-1)
    perm[n] = pd.Series(r.importances_mean, index=FEATURES)
perm_df = pd.DataFrame(perm)
perm_mean = perm_df.mean(axis=1).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
imp_top = imp_dt.tail(12)
ax[0].barh(imp_top.index, imp_top.values, color="#4c78a8"); ax[0].set_title("Decision Tree (Gini) importance, top 12")
ax[1].barh(perm_mean.index, perm_mean.values, color="#f58518")
ax[1].set_title("Permutation importance (mean of 4 models)")
plt.tight_layout(); plt.show()

display(perm_df.round(4).assign(Mean=perm_df.mean(axis=1).round(4)).sort_values("Mean", ascending=False))

Decision Tree rules (readable by a clinician):

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(dt_pipe.named_steps["clf"], feature_names=list(enc_names), class_names=["No disease", "Disease"],
          filled=True, max_depth=3, fontsize=8, proportion=True)
plt.show()

## Step 16: Strengths, weaknesses and limitations

| Model | Strengths | Weaknesses |
|---|---|---|
| Decision Tree | Easy-to-read rules, handles mixed features, fast | Unstable on 300 rows (a few patients can change the tree), can overfit |
| Naive Bayes | Works well with little data, very fast, few parameters | Independence assumption is not true for clinical features |
| KNN | Simple, no training, good on small data | Sensitive to k and scaling, not interpretable, slow on big data |
| ANN (MLP) | Learns non-linear interactions | Needs tuning, black box, easily overfits when data is small |

**Limitations of this dataset/study**
- Only 303 patients from one clinic (Cleveland), so results may not generalise.
- The test set is tiny (61), so differences of 1-3 points between models are within noise; use the repeated cross-validation table to compare.
- Features include clinical tests (ECG, fluoroscopy, thalassemia test) that are not available in simple screening, unlike the 70,000-record dataset.
- Target is angiographic disease status, binarised from 0-4.
- This is a risk-estimation aid, **not** a medical diagnosis.

## Step 17: Choose the best model
The code picks the model with the highest mean cross-validated F1-score (ties are broken by recall, which matters for screening).

In [ ]:
rank = cv_table[["_f1", "_rec", "_acc"]].sort_values(["_f1", "_rec"], ascending=False)
display(rank.rename(columns={"_f1": "CV F1", "_rec": "CV Recall", "_acc": "CV Accuracy"}).round(4))
best_name = rank.index[0]
print("Best model on repeated cross-validation:", best_name)

## Step 18: Conceptual prediction system (demo)
Enter a patient's values. The best model gives the risk probability and the Decision Tree gives a second opinion.

In [ ]:
# the best model was already fitted in Step 11 on the training split; refit on all data for deployment
final_best = make_pipe(models[best_name].named_steps["clf"]).fit(X, y)
final_tree = make_pipe(DecisionTreeClassifier(random_state=SEED, **{k.replace("clf__", ""): v
                       for k, v in dt_search.best_params_.items()})).fit(X, y)

def predict_patient(age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal):
    """sex 1=male 0=female | cp 1-4 | fbs 0/1 | restecg 0-2 | exang 0/1 | slope 1-3 | ca 0-3 | thal 3,6,7"""
    row = pd.DataFrame([[age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]],
                       columns=["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
                                "exang", "oldpeak", "slope", "ca", "thal"])[FEATURES]
    p_best = final_best.predict_proba(row)[0, 1]
    p_tree = final_tree.predict_proba(row)[0, 1]
    level = "LOW" if p_best < 0.35 else ("MODERATE" if p_best < 0.65 else "HIGH")
    print(f"Best model ({best_name}) risk probability: {p_best:.1%}")
    print(f"Decision Tree risk probability       : {p_tree:.1%}")
    print(f"Risk level                           : {level}")
    print("Predicted class                      :", "Heart disease" if p_best >= 0.5 else "No heart disease")
    print("(Screening aid only, not a medical diagnosis. Consult a doctor.)")

predict_patient(age=62, sex=1, cp=4, trestbps=150, chol=260, fbs=0, restecg=2,
                thalach=110, exang=1, oldpeak=2.5, slope=2, ca=2, thal=7)

## Step 19: Save and download the results

In [ ]:
from google.colab import files
out = results_df.round(4)
out.to_csv("uci_model_comparison_results.csv")
cv_table.drop(columns=["_acc", "_f1", "_rec"]).to_csv("uci_cross_validation_results.csv")
df.to_csv("uci_heart_cleaned.csv", index=False)
files.download("uci_model_comparison_results.csv")
files.download("uci_cross_validation_results.csv")
files.download("uci_heart_cleaned.csv")